In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from pathlib import Path

user_secrets = UserSecretsClient()

import time


def get_secret_with_retry(label, retries=5, delay=3):
    # Kaggle Secrets サービスが一時的に 400/ConnectionError を返すことがあるためリトライする
    for attempt in range(retries):
        try:
            return user_secrets.get_secret(label)
        except Exception as e:
            print(f"get_secret('{label}') 失敗 (試行 {attempt + 1}/{retries}): {e}")
            if attempt == retries - 1:
                raise
            time.sleep(delay)


import json

# Secrets は private Kaggle Dataset (voice-clone-secrets) から読む。
# API push (kernels_push) では手動アタッチした Kaggle Secrets が引き継がれないため、
# データセット経由で渡す。データセットが無ければ get_secret にフォールバックする。
_SECRETS_PATH = Path("/kaggle/input/voice-clone-secrets/secrets.json")
_secrets = json.loads(_SECRETS_PATH.read_text()) if _SECRETS_PATH.exists() else None


def secret(label):
    if _secrets is not None and label in _secrets:
        return _secrets[label]
    return get_secret_with_retry(label)


BASE_URL = "https://neglegere.com"
EMAIL = secret("email")
PASSWORD = secret("password")
SAVE_DIR = Path("/kaggle/working/reference_audio")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_TOKEN"] = secret("hf_token")

R2_ACCOUNT_ID = secret("r2_AccountID")
R2_ACCESS_KEY = secret("r2_accesskey")
R2_SECRET_KEY = secret("r2_secretkey")
R2_BUCKET_NAME = secret("r2_bucket_name")
R2_ENDPOINT = f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com"

WEBHOOK_URL = secret("webhook_url")


# 通知名は実行時の GPU 有無から決める。
# これにより CPU 版 / T4 版でノートブックのコードを完全に同一に保てる
# （2ファイルの差分は Kaggle metadata の accelerator 設定だけになる）。
HAS_GPU = Path("/proc/driver/nvidia/version").exists()
NOTEBOOK_NAME = "voice-clone-task-by-T4" if HAS_GPU else "voice-clone-task"


def notify_slack(text: str):
    import requests
    try:
        requests.post(WEBHOOK_URL, json={"text": text}, timeout=15)
    except Exception as e:
        print("Slack通知に失敗:", e)


# --- pending を先に確認し、0件なら以降の重い処理をスキップする ---
import boto3
import json as _json


def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    res = s3.get_object(Bucket=R2_BUCKET_NAME, Key=key)
    return _json.loads(res["Body"].read().decode("utf-8"))


_pending = fetch_pending_voice_clones()
HAS_WORK = bool(_pending and _pending.get("items"))
print(f"pending: {len(_pending.get('items', [])) if _pending else 0} 件")
if not HAS_WORK:
    print("0件のため pip install / モデル処理をスキップします")


In [ ]:
# torch/soundfile/transformers/accelerate は Kaggle に同梱済みのため入れない
# pending がある時だけ qwen-tts を入れる（0件なら pip を走らせない）
if HAS_WORK:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "qwen-tts", "librosa", "scipy"],
        check=True,
    )

import os
import signal
import time

import torch
import soundfile as sf
from pathlib import Path


class GenerationTimeout(Exception):
    pass


class GenerationRunaway(Exception):
    """EOS を出せず max_new_tokens まで走り切った（暴走した）ことを示す。"""


# --- 生成の暴走対策 ----------------------------------------------------------
# Qwen3-TTS-12Hz は 1 秒 = 12 codec フレーム。1 フレームごとに talker(28層) を
# 1 step、さらに code_predictor(num_code_groups=16) を 15 step 回すため、
# 生成フレーム数がそのまま実時間に直結する。
# EOS(codec_eos_token_id) を出し損なうと max_new_tokens（ライブラリ既定 2048
# = 約 170 秒ぶんの音声）まで走り続けるので、短いテキストでも 10 分を超える。
# テキスト長から上限フレーム数を絞り、暴走を「短時間で検知して作り直す」形にする。
CODEC_FPS = 12          # Qwen3-TTS-12Hz のフレームレート
CHARS_PER_SEC = 7.0     # 日本語の読み上げ速度のざっくり見積り
LENGTH_MARGIN = 3.0     # 見積りに対する余裕
MIN_NEW_TOKENS = 192    # 約 16 秒。短文でもこれ以下には絞らない
MAX_NEW_TOKENS = 2048   # ライブラリ既定と同じ天井

# 1 回の generate に許す秒数。max_new_tokens を絞ってあるので 10 分は不要。
ATTEMPT_TIMEOUT_SEC = 240

# 暴走・失敗時に条件を変えて再試行する順序（1 回目はライブラリ既定のまま）。
# 既定は do_sample=True / temperature=0.9 なので、引き直すだけでも成功しうる。
# 2 回目以降は決定性寄りに振り、繰り返しにペナルティを強める。
GENERATE_ATTEMPTS = [
    {},
    {"temperature": 0.7, "top_k": 20, "repetition_penalty": 1.15},
    {"temperature": 0.6, "top_k": 10, "repetition_penalty": 1.25},
]


def estimate_max_new_tokens(text: str) -> int:
    """テキスト長から生成フレーム数の上限を見積もる。"""
    est_frames = len(text) / CHARS_PER_SEC * CODEC_FPS * LENGTH_MARGIN
    return int(min(MAX_NEW_TOKENS, max(MIN_NEW_TOKENS, est_frames)))


def _free_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


_qwen_model_cache = {}


def _get_qwen_model(model_name):
    # モデルはプロセス内で1回だけロードして使い回す（item ごとの再ロードを避ける）
    if model_name not in _qwen_model_cache:
        from qwen_tts import Qwen3TTSModel

        device = "cuda" if torch.cuda.is_available() else "cpu"
        _qwen_model_cache[model_name] = Qwen3TTSModel.from_pretrained(
            model_name,
            device_map="cuda:0" if device == "cuda" else "cpu",
            dtype=torch.float16 if device == "cuda" else torch.float32,
        )
        print(f"✅ Model loaded on {device}")
    return _qwen_model_cache[model_name]


def _generate_once(model, text, ref_audio_path, ref_text, max_new_tokens, overrides):
    """generate_voice_clone を 1 回だけ実行する。(wav, sr, 所要秒) を返す。"""

    def _on_generate_timeout(signum, frame):
        raise GenerationTimeout(
            f"generate_voice_clone が {ATTEMPT_TIMEOUT_SEC} 秒を超えたため中断しました"
        )

    old_handler = signal.signal(signal.SIGALRM, _on_generate_timeout)
    signal.alarm(ATTEMPT_TIMEOUT_SEC)
    started = time.monotonic()
    try:
        wavs, sr = model.generate_voice_clone(
            text=text,
            language="Japanese",
            ref_audio=ref_audio_path,
            ref_text=ref_text,
            max_new_tokens=max_new_tokens,
            **overrides,
        )
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    return wavs[0], sr, time.monotonic() - started


def generate_qwen_voice(
    text: str,
    ref_audio_path: str,
    ref_text: str,
    output_path: str = "/kaggle/working/qwen_generated.wav",
    model_name: str = "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
):
    """
    Qwen3-TTSを使用してボイスクローン音声を生成するメソッド。

    テキスト長から生成フレーム数の上限を決め、暴走（EOS 未出力で上限に張り付く）
    を検知したら条件を変えて再試行する。全試行が失敗した場合のみ例外を投げる。

    Args:
        text (str): 生成する音声のテキスト
        ref_audio_path (str): 参照音声（WAVファイル）のパス
        ref_text (str): 参照音声の読み上げ内容
        output_path (str): 生成された音声の保存先
        model_name (str): 使用するQwenモデル

    Returns:
        str: 生成された音声ファイルのパス

    Raises:
        GenerationTimeout: 全試行で生成できなかった場合
    """

    # モデルは1回だけロードして使い回す
    model = _get_qwen_model(model_name)

    max_new_tokens = estimate_max_new_tokens(text)
    cap_sec = max_new_tokens / CODEC_FPS
    print(
        f"generate: {len(text)}文字 -> max_new_tokens={max_new_tokens} "
        f"(上限 {cap_sec:.0f}秒ぶん)"
    )

    last_error = None

    for attempt, overrides in enumerate(GENERATE_ATTEMPTS, start=1):
        label = f"試行 {attempt}/{len(GENERATE_ATTEMPTS)}"
        try:
            wav, sr, elapsed = _generate_once(
                model, text, ref_audio_path, ref_text, max_new_tokens, overrides
            )
        except (GenerationTimeout, RuntimeError) as e:
            # RuntimeError には CUDA OOM も含まれる。キャッシュを空けて条件を変えて再試行。
            last_error = e
            print(f"{label}: 失敗 ({type(e).__name__}: {e})")
            _free_gpu()
            continue

        duration = len(wav) / sr

        # 上限に張り付いた = EOS を出せていない。音声として使えないので捨てる。
        if duration >= cap_sec * 0.95:
            last_error = GenerationRunaway(
                f"生成 {duration:.1f}秒 が上限 {cap_sec:.0f}秒 に到達（EOS 未出力）"
            )
            print(f"{label}: 暴走 ({last_error}) 所要 {elapsed:.0f}秒")
            _free_gpu()
            continue

        print(f"✅ {label} 成功: {duration:.1f}秒 の音声を {elapsed:.0f}秒 で生成")

        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(str(output_path), wav, sr)
        print(f"✅ Generated voice saved: {output_path}")

        return str(output_path)

    raise GenerationTimeout(
        f"{len(GENERATE_ATTEMPTS)} 回試しましたが生成できませんでした: {last_error}"
    )


In [ ]:
import requests
from pathlib import Path

def login() -> str:
    url = f"{BASE_URL}/api/auth/login"
    headers = {
        "accept": "application/json",
        "Content-Type": "application/json",
    }
    payload = {
        "email": EMAIL,
        "password": PASSWORD,
    }

    res = requests.post(url, headers=headers, json=payload, timeout=30)
    res.raise_for_status()

    data = res.json()

    token = data.get("token")

    return token

def get_pending_items(token: str) -> list[dict]:
    url = f"{BASE_URL}/api/voice_clones/pending"
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.get(url, headers=headers, timeout=30)
    res.raise_for_status()

    data = res.json()

    items = data.get("items", [])

    return items

def start_voice_clone(token: str, voice_clone_id: str) -> bool:
    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/start"
    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, timeout=30)

    if res.status_code == 409:
        return False

    res.raise_for_status()

    return res.json().get("status") == "generating"

def download_reference_audio(token: str, voice_clone_id: str, file_name: str | None = None) -> Path:
    url = f"{BASE_URL}/api/reference_audio/{voice_clone_id}/file"
    headers = {
        "accept": "audio/wav",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, data="", stream=True, timeout=60)
    res.raise_for_status()

    output_name = Path(file_name).name

    output_path = SAVE_DIR / output_name

    with open(output_path, "wb") as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

    return output_path

def upload_completed_voice_clone(
    token: str,
    voice_clone_id: str,
    audio_file_path,
):
    audio_file_path = Path(audio_file_path)

    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/complete"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    with open(audio_file_path, "rb") as f:
        files = {
            "file": (audio_file_path.name, f, "audio/wav")
        }

        res = requests.post(
            url,
            headers=headers,
            files=files,
            timeout=60,
        )

    res.raise_for_status()

    data = res.json()

    if not data.get("ok"):
        raise Exception(f"APIエラー: {data}")

    return data


def fail_voice_clone(token: str, voice_clone_id: str):
    url = f"{BASE_URL}/api/voice_clones/{voice_clone_id}/fail"

    headers = {
        "accept": "application/json",
        "Authorization": f"Bearer {token}",
    }

    res = requests.post(url, headers=headers, timeout=30)
    res.raise_for_status()

    return res.json()


In [ ]:
import boto3
import json

def fetch_pending_voice_clones(key: str = "pending.json"):
    s3 = boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT,
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )

    res = s3.get_object(
        Bucket=R2_BUCKET_NAME,
        Key=key,
    )

    return json.loads(res["Body"].read().decode("utf-8"))

print(fetch_pending_voice_clones())

In [ ]:
import shutil
import time

def main():

    while True:
        items = fetch_pending_voice_clones()

        if not items or not items.get("items"):
            print("pending は 0 件です")
            break

        for item in items["items"]:
            voice_clone_id = item["id"]
            referenceAudioPath = item["referenceAudioPath"]
            token = login()

            if not start_voice_clone(token, voice_clone_id):
                print(f"skip id={voice_clone_id}: ほかの処理が開始済みです")
                continue

            Path("/kaggle/working/reference_audio").mkdir(parents=True, exist_ok=True)
            Path("/kaggle/working/clone").mkdir(parents=True, exist_ok=True)

            print(f"downloading id={voice_clone_id} ...")
            output_path = download_reference_audio(
                token, 
                voice_clone_id, 
                referenceAudioPath
            )
            print(f"saved: {output_path}")

            # 音声生成（暴走・タイムアウト時は再試行し、全滅ならスキップ）
            try:
                output_file = generate_qwen_voice(
                    text=item["desiredText"],
                    ref_audio_path=str(output_path),
                    ref_text=item["recordedText"],
                    output_path=Path("/kaggle/working/clone/") / Path(referenceAudioPath).name,
                )
            except GenerationTimeout as e:
                print(f"skip id={voice_clone_id}: {e}")
                notify_slack(
                    f":warning: {NOTEBOOK_NAME}: id={voice_clone_id} の生成に失敗したためスキップしました\n```\n{e}\n```"
                )
                try:
                    fail_voice_clone(login(), voice_clone_id)
                except Exception as fe:
                    print(f"fail 通知に失敗: {fe}")
                shutil.rmtree("/kaggle/working/reference_audio", ignore_errors=True)
                shutil.rmtree("/kaggle/working/clone", ignore_errors=True)
                continue

            upload_completed_voice_clone(
                token,
                voice_clone_id,
                output_file
            )
        
            print("生成されたファイル:", output_file)
        
            shutil.rmtree("/kaggle/working/reference_audio")
            shutil.rmtree("/kaggle/working/clone")

    print("\n完了")

import traceback

if HAS_WORK:
    try:
        main()
    except Exception:
        tb = traceback.format_exc()
        print(tb)
        notify_slack(
            f":rotating_light: {NOTEBOOK_NAME} が失敗しました\n```\n{tb[-3000:]}\n```"
        )
        raise
else:
    print("pending 0件のため main() を実行しません")
